In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import os
import shutil
import random

SOURCE_DIR = "ISL_CSLRT_Corpus/Frames_Word_Level"
TARGET_DIR = "dataset"

TRAIN_DIR = os.path.join(TARGET_DIR, "train")
VAL_DIR   = os.path.join(TARGET_DIR, "val")

split_ratio = 0.8

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

for label in os.listdir(SOURCE_DIR):

    label_path = os.path.join(SOURCE_DIR, label)

    if not os.path.isdir(label_path):
        continue

    images = os.listdir(label_path)
    random.shuffle(images)

    split_idx = int(len(images) * split_ratio)

    train_imgs = images[:split_idx]
    val_imgs   = images[split_idx:]

    os.makedirs(os.path.join(TRAIN_DIR, label), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, label), exist_ok=True)

    # Copy train images
    for img in train_imgs:
        src = os.path.join(label_path, img)
        dst = os.path.join(TRAIN_DIR, label, img)
        shutil.copy(src, dst)

    # Copy val images
    for img in val_imgs:
        src = os.path.join(label_path, img)
        dst = os.path.join(VAL_DIR, label, img)
        shutil.copy(src, dst)

    print(f"{label}: Train {len(train_imgs)} | Val {len(val_imgs)}")

print("\n✅ Dataset split complete!")

A LOT: Train 3 | Val 1
ABUSE: Train 4 | Val 1
AFRAID: Train 4 | Val 2
AGREE: Train 5 | Val 2
ALL: Train 3 | Val 1
ANGRY: Train 8 | Val 2
ANYTHING: Train 3 | Val 1
APPRECIATE: Train 4 | Val 1
BAD: Train 4 | Val 1
BEAUTIFUL: Train 4 | Val 1
BECOME: Train 4 | Val 1
BED: Train 5 | Val 2
BORED: Train 4 | Val 2
BRING: Train 5 | Val 2
CHAT: Train 4 | Val 2
CLASS: Train 4 | Val 1
COLD: Train 5 | Val 2
COLLEGE_SCHOOL: Train 14 | Val 4
COMB: Train 5 | Val 2
COME: Train 4 | Val 2
CONGRATULATIONS: Train 5 | Val 2
CRYING: Train 10 | Val 3
DARE: Train 4 | Val 2
DIFFERENCE: Train 4 | Val 1
DILEMMA: Train 4 | Val 2
DISAPPOINTED: Train 4 | Val 2
DO: Train 9 | Val 3
DON'T CARE: Train 2 | Val 1
ENJOY: Train 3 | Val 1
FAVOUR: Train 3 | Val 1
FEVER: Train 5 | Val 2
FINE: Train 4 | Val 2
FOOD: Train 18 | Val 5
FREE: Train 5 | Val 2
FRIEND: Train 4 | Val 2
FROM: Train 5 | Val 2
GO: Train 7 | Val 2
GOOD: Train 5 | Val 2
GRATEFUL: Train 4 | Val 2
HAD: Train 3 | Val 1
HAPPENED: Train 4 | Val 1
HAPPY: Train 4 | 

In [3]:
import os
import cv2
import random
import pandas as pd
from tqdm import tqdm
import albumentations as A

TRAIN_DIR = "dataset/train"
OUTPUT_DIR = "dataset/train_aug"

TARGET_PER_CLASS = 50   # you wanted 50 per class

os.makedirs(OUTPUT_DIR, exist_ok=True)

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Rotate(limit=25, p=0.5),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=20, p=0.5),
    A.GaussianBlur(p=0.3),
])

data = []

for label in os.listdir(TRAIN_DIR):

    label_path = os.path.join(TRAIN_DIR, label)
    save_path  = os.path.join(OUTPUT_DIR, label)

    os.makedirs(save_path, exist_ok=True)

    images = os.listdir(label_path)

    print(f"\nProcessing {label} ({len(images)} images)")

    # Case 1: Reduce if too many
    if len(images) >= TARGET_PER_CLASS:

        selected = random.sample(images, TARGET_PER_CLASS)

        for img_name in selected:
            img = cv2.imread(os.path.join(label_path, img_name))
            out_path = os.path.join(save_path, img_name)

            cv2.imwrite(out_path, img)
            data.append([out_path, label])

    # Case 2: Augment if too few
    else:
        count = 0

        # Save originals
        for img_name in images:
            img = cv2.imread(os.path.join(label_path, img_name))
            out_path = os.path.join(save_path, f"orig_{img_name}")

            cv2.imwrite(out_path, img)
            data.append([out_path, label])
            count += 1

        # Generate more
        while count < TARGET_PER_CLASS:

            img_name = random.choice(images)
            img = cv2.imread(os.path.join(label_path, img_name))

            aug_img = augment(image=img)["image"]

            new_name = f"aug_{count}.jpg"
            out_path = os.path.join(save_path, new_name)

            cv2.imwrite(out_path, aug_img)
            data.append([out_path, label])

            count += 1

print("\n✅ Augmentation + Balancing Done")

C:\Users\harsh\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\harsh\AppData\Roaming\Python\Python310\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)



Processing A LOT (3 images)

Processing ABUSE (4 images)

Processing AFRAID (4 images)

Processing AGREE (5 images)

Processing ALL (3 images)

Processing ANGRY (8 images)

Processing ANYTHING (3 images)

Processing APPRECIATE (4 images)

Processing BAD (4 images)

Processing BEAUTIFUL (4 images)

Processing BECOME (4 images)

Processing BED (5 images)

Processing BORED (4 images)

Processing BRING (5 images)

Processing CHAT (4 images)

Processing CLASS (4 images)

Processing COLD (5 images)

Processing COLLEGE_SCHOOL (14 images)

Processing COMB (5 images)

Processing COME (4 images)

Processing CONGRATULATIONS (5 images)

Processing CRYING (10 images)

Processing DARE (4 images)

Processing DIFFERENCE (4 images)

Processing DILEMMA (4 images)

Processing DISAPPOINTED (4 images)

Processing DO (9 images)

Processing DON'T CARE (2 images)

Processing ENJOY (3 images)

Processing FAVOUR (3 images)

Processing FEVER (5 images)

Processing FINE (4 images)

Processing FOOD (18 images)

P

In [4]:
df_train = pd.DataFrame(data, columns=["image_path", "label"])

df_train.to_csv("train.csv", index=False)

print(df_train.head())
print("Total samples:", len(df_train))

                                   image_path  label
0  dataset/train_aug\A LOT\orig_A LOT (1).jpg  A LOT
1  dataset/train_aug\A LOT\orig_A LOT (2).jpg  A LOT
2  dataset/train_aug\A LOT\orig_A LOT (4).jpg  A LOT
3           dataset/train_aug\A LOT\aug_3.jpg  A LOT
4           dataset/train_aug\A LOT\aug_4.jpg  A LOT
Total samples: 5700


In [5]:
VAL_DIR = "dataset/val"

val_data = []

for label in os.listdir(VAL_DIR):

    label_path = os.path.join(VAL_DIR, label)

    for img_name in os.listdir(label_path):

        img_path = os.path.join(label_path, img_name)
        val_data.append([img_path, label])

df_val = pd.DataFrame(val_data, columns=["image_path", "label"])

df_val.to_csv("val.csv", index=False)

print(df_val.head())
print("Total val samples:", len(df_val))

                          image_path   label
0    dataset/val\A LOT\A LOT (3).jpg   A LOT
1    dataset/val\ABUSE\ABUSE (3).jpg   ABUSE
2  dataset/val\AFRAID\AFRAID (4).jpg  AFRAID
3  dataset/val\AFRAID\AFRAID (6).jpg  AFRAID
4    dataset/val\AGREE\AGREE (3).jpg   AGREE
Total val samples: 257


In [6]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.ColorJitter(0.3,0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])